In [ ]:
import pandas as pd

In [ ]:
from src.load.internal.db.database_session import get_engine

engine = get_engine()

In [ ]:
pd.read_sql("SELECT COUNT(*) from visits", engine)

In [ ]:
pd.read_sql("SELECT MIN(visit_date) from visits", engine)

In [ ]:
pd.read_sql("SELECT MAX(visit_date) from visits", engine)

In [ ]:
# visits_count by styles
pd.read_sql("""
SELECT styles.name, COUNT(*) AS visits_count FROM visits 
INNER JOIN styles ON styles.id = visits.style_id
GROUP BY styles.name
ORDER BY visits_count DESC;
""", engine)

In [ ]:
# visits_count by teachers and styles
pd.read_sql("""
SELECT styles.name, CONCAT(teachers.last_name, ' ', teachers.name) AS teacher_full_name, 
COUNT(*) AS visits_count FROM visits 
INNER JOIN styles ON styles.id = visits.style_id
INNER JOIN teachers ON teachers.id = visits.teacher_id
GROUP BY styles.name, teacher_full_name
ORDER BY visits_count DESC;
""", engine)

In [ ]:
# visits_count by weekdays
df_by_weekdays = pd.read_sql("""
SELECT  
    weekday,
    CASE weekday
        WHEN 0 THEN 'Monday'
        WHEN 1 THEN 'Tuesday'
        WHEN 2 THEN 'Wednesday'
        WHEN 3 THEN 'Thursday'
        WHEN 4 THEN 'Friday'
        WHEN 5 THEN 'Saturday'
        WHEN 6 THEN 'Sunday'
    END AS weekday_name,
    COUNT(*) AS visits_count 
FROM visits
GROUP BY weekday
ORDER BY weekday;
""", engine)
df_by_weekdays

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(
    df_by_weekdays["weekday_name"],
    df_by_weekdays["visits_count"]
)

plt.title("Visits by weekday")
plt.xlabel("Weekday")
plt.ylabel("Visits")

plt.show()

In [ ]:
# visits_count by time
df_by_hours = pd.read_sql("""
SELECT
    CONCAT(FLOOR(minutes_begin / 60), ':00') AS hour,
    COUNT(*) AS visits_count
FROM visits
GROUP BY hour
ORDER BY hour;
""", engine)
df_by_hours

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(
    df_by_hours["hour"],
    df_by_hours["visits_count"]
)

plt.title("Visits by hour")
plt.xlabel("Hour")
plt.ylabel("Visits")

plt.show()

In [ ]:
# top teachers
df_by_teachers = pd.read_sql("""
SELECT
    CONCAT(teachers.last_name, ' ', teachers.name) AS teacher_full_name,
    COUNT(*) visits,
    RANK() OVER(
        ORDER BY COUNT(*) DESC
    ) teacher_rank
FROM visits
INNER JOIN teachers ON teachers.id = visits.teacher_id
GROUP BY teachers.last_name, teachers.name;
""", engine)
df_by_teachers

In [ ]:
df_by_teachers_sorted = (
    df_by_teachers
    .sort_values("visits", ascending=False)
    .head(25)
    .sort_values("visits")
)

plt.figure(figsize=(16, 20))

plt.barh(
    df_by_teachers_sorted["teacher_full_name"],
    df_by_teachers_sorted["visits"]
)

plt.title("Visits by teachers")
plt.xlabel("Visits")
plt.ylabel("Teacher")

plt.tight_layout()
plt.show()

In [ ]:
# limited group_accounts
pd.read_sql("""
SELECT styles.id, styles.name, ROUND(SUM(visits.lesson_cost), 2) AS amount
FROM visits
INNER JOIN styles ON styles.id = visits.style_id
WHERE visits.group_account_is_unlimited = false
GROUP BY styles.name, styles.id
ORDER BY amount DESC;
""", engine)

In [ ]:
# group_singles
pd.read_sql("""
SELECT styles.id, styles.name, ROUND(SUM(group_singles.cost), 2) AS amount
FROM group_singles
INNER JOIN styles ON styles.id = group_singles.style_id
GROUP BY styles.name, styles.id
ORDER BY amount DESC;
""", engine)

In [ ]:
# list of unlimited group_accounts
pd.read_sql("""
SELECT 
group_account_id, 
COUNT(*) AS visits_count,
MAX(group_account_cost) / COUNT(*) AS lesson_cost
FROM visits
WHERE group_account_is_unlimited = true
GROUP BY group_account_id
""", engine)

In [ ]:
# revenue by unlimited group_account
pd.read_sql("""
WITH unlimited_abonements AS ( 
    SELECT 
    group_account_id, 
    MAX(group_account_cost) / COUNT(*) AS lesson_cost
    FROM visits
    WHERE group_account_is_unlimited = true
    GROUP BY group_account_id
)

SELECT styles.id, styles.name, ROUND(SUM(unlimited_abonements.lesson_cost), 2) AS amount
FROM visits
JOIN unlimited_abonements ON unlimited_abonements.group_account_id = visits.group_account_id
JOIN styles ON styles.id = visits.style_id
WHERE visits.group_account_is_unlimited = true
GROUP BY styles.id, styles.name
ORDER BY amount DESC;

""", engine)

In [ ]:
# revenue by stytes
df_amount_by_styles = pd.read_sql("""

WITH unlimited_abonements AS ( 
    SELECT 
    group_account_id, 
    MAX(group_account_cost) / COUNT(*) AS lesson_cost
    FROM visits
    WHERE group_account_is_unlimited = true
    GROUP BY group_account_id
),
revenue AS (
    SELECT styles.id, styles.name, ROUND(SUM(visits.lesson_cost), 2)::numeric AS amount
    FROM visits
    INNER JOIN styles ON styles.id = visits.style_id
    WHERE visits.group_account_is_unlimited = false
    GROUP BY styles.name, styles.id

    UNION ALL
    
    SELECT styles.id, styles.name, ROUND(SUM(unlimited_abonements.lesson_cost), 2)::numeric AS amount
    FROM visits
    JOIN unlimited_abonements ON unlimited_abonements.group_account_id = visits.group_account_id
    JOIN styles ON styles.id = visits.style_id
    WHERE visits.group_account_is_unlimited = true
    GROUP BY styles.id, styles.name

    UNION ALL

    SELECT styles.id, styles.name, ROUND(SUM(group_singles.cost), 2)::numeric AS amount
    FROM group_singles
    INNER JOIN styles ON styles.id = group_singles.style_id
    GROUP BY styles.name, styles.id
)

SELECT styles.id, styles.name, ROUND(SUM(revenue.amount), 2)::numeric AS amount
FROM revenue 
INNER JOIN styles ON styles.id = revenue.id
GROUP BY styles.id, styles.name
ORDER BY amount DESC;
""", engine)
df_amount_by_styles

In [ ]:
plt.figure(figsize=(16,4))
plt.bar(
    df_amount_by_styles["name"],
    df_amount_by_styles["amount"]
)

plt.title("Revenue by styles")
plt.xlabel("Style")
plt.xticks(rotation=90)
plt.ylabel("Revenue")

plt.show()

In [ ]:
# lessons by dates grouped
df_lessons_by_dates = pd.read_sql("""

WITH unlimited_abonements AS ( 
    SELECT 
    group_account_id, 
    MAX(group_account_cost) / COUNT(*) AS lesson_cost
    FROM visits
    WHERE group_account_is_unlimited = true
    GROUP BY group_account_id
),
all_visits AS (
    SELECT visits.id, visits.minutes_begin, visits.minutes_end, visits.group_id, visits.style_id, visits.teacher_id, visits.client_id,
    visits.visit_date, visits.month, visits.weekday, visits.is_weekend, visits.season, visits.quarter, visits.lesson_cost AS cost
    FROM visits
    WHERE visits.group_account_is_unlimited = false

    UNION ALL
    
    SELECT visits.id, visits.minutes_begin, visits.minutes_end, visits.group_id, visits.style_id, visits.teacher_id, visits.client_id,
    visits.visit_date, visits.month, visits.weekday, visits.is_weekend, visits.season, visits.quarter, 
    unlimited_abonements.lesson_cost AS cost
    FROM visits
    INNER JOIN unlimited_abonements ON unlimited_abonements.group_account_id = visits.group_account_id
    WHERE visits.group_account_is_unlimited = true

    UNION ALL

    SELECT group_singles.id, group_singles.minutes_begin, group_singles.minutes_end, group_singles.group_id, group_singles.style_id, group_singles.teacher_id, group_singles.client_id,
    group_singles.visit_date, group_singles.month, group_singles.weekday, group_singles.is_weekend, group_singles.season, group_singles.quarter,
    group_singles.cost
    FROM group_singles
)

SELECT all_visits.visit_date, all_visits.minutes_begin, all_visits.minutes_end, all_visits.group_id, all_visits.style_id, styles.name AS style_name, all_visits.teacher_id, CONCAT(teachers.last_name, ' ', teachers.name) AS teacher_full_name,
    all_visits.month, all_visits.weekday, all_visits.is_weekend, all_visits.season, all_visits.quarter, 
    ROUND(SUM(all_visits.cost), 2)::numeric AS amount, COUNT(*) AS visits_count
FROM all_visits 
INNER JOIN styles ON styles.id = all_visits.style_id
INNER JOIN teachers ON teachers.id = all_visits.teacher_id
GROUP BY all_visits.visit_date, all_visits.minutes_begin, all_visits.minutes_end, all_visits.group_id, all_visits.style_id, all_visits.teacher_id,
    all_visits.month, all_visits.weekday, all_visits.is_weekend, all_visits.season, all_visits.quarter, style_name, teacher_full_name
ORDER BY all_visits.visit_date;
""", engine)
df_lessons_by_dates

In [ ]:
df_lessons_by_dates.to_parquet("./df_lessons_by_dates.parquet", index=False)

In [ ]:
df_cumulative_revenue = pd.read_sql("""

WITH unlimited_abonements AS ( 
    SELECT 
    group_account_id, 
    MAX(group_account_cost) / COUNT(*) AS lesson_cost
    FROM visits
    WHERE group_account_is_unlimited = true
    GROUP BY group_account_id
),
all_visits AS (
    SELECT visits.id, visits.minutes_begin, visits.minutes_end, visits.group_id, visits.style_id, visits.teacher_id, visits.client_id,
    visits.visit_date, visits.month, visits.weekday, visits.is_weekend, visits.season, visits.quarter, visits.lesson_cost AS cost
    FROM visits
    WHERE visits.group_account_is_unlimited = false

    UNION ALL
    
    SELECT visits.id, visits.minutes_begin, visits.minutes_end, visits.group_id, visits.style_id, visits.teacher_id, visits.client_id,
    visits.visit_date, visits.month, visits.weekday, visits.is_weekend, visits.season, visits.quarter, 
    unlimited_abonements.lesson_cost AS cost
    FROM visits
    INNER JOIN unlimited_abonements ON unlimited_abonements.group_account_id = visits.group_account_id
    WHERE visits.group_account_is_unlimited = true

    UNION ALL

    SELECT group_singles.id, group_singles.minutes_begin, group_singles.minutes_end, group_singles.group_id, group_singles.style_id, group_singles.teacher_id, group_singles.client_id,
    group_singles.visit_date, group_singles.month, group_singles.weekday, group_singles.is_weekend, group_singles.season, group_singles.quarter,
    group_singles.cost
    FROM group_singles
)

SELECT
    visit_date,
    SUM(cost) AS daily_revenue,
    SUM(SUM(cost))
        OVER (
            ORDER BY visit_date
        ) AS cumulative_revenue
FROM all_visits
GROUP BY visit_date
ORDER BY visit_date;
""", engine)
df_cumulative_revenue

In [ ]:
plt.figure(figsize=(12,5))

plt.bar(
    df_cumulative_revenue["visit_date"],
    df_cumulative_revenue["daily_revenue"]
)

plt.title("Daily revenue")
plt.xlabel("Date")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    df_cumulative_revenue["visit_date"],
    df_cumulative_revenue["cumulative_revenue"]
)

plt.title("Cumulative revenue")
plt.xlabel("Date")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()